[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/galvezam/eia-data-pipeline/blob/main/processing/natural_gas_crude_processing.ipynb)

# Natural Gas & Crude Oil Processing
**EIA Data Pipeline**

Datasets:
- `natural_gas_*.csv` — Monthly marketed production by state (MMcf)
- `natural_gas_trade_*.csv` — Annual state-level exports, imports, and interstate movements (MMcf)
- `crude_oil_imports_*.csv` — Monthly crude oil imports by country of origin → refinery state (thousand barrels)

Pipeline steps: **Clean → Transform → Analyze → Upload Parquet to S3**

## 0. Environment Setup & Configuration

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# Install Spark with Hadoop as the underlying scheduler
!wget -q https://downloads.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar xf spark-3.5.8-bin-hadoop3.tgz
!ls -l

os.environ["SPARK_HOME"] = "spark-3.5.8-bin-hadoop3"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

!pip install -q findspark
import findspark
findspark.init()
print("Spark environment ready.")

# Install boto3
!pip install boto3

total 1175932
drwxr-xr-x  1 root root      4096 Apr 16 13:28 sample_data
drwxr-xr-x 13 1001 1001      4096 Jan 12 04:31 spark-3.5.8-bin-hadoop3
-rw-r--r--  1 root root 401378872 Jan 12 04:34 spark-3.5.8-bin-hadoop3.tgz
-rw-r--r--  1 root root 401378872 Jan 12 04:34 spark-3.5.8-bin-hadoop3.tgz.1
-rw-r--r--  1 root root 401378872 Jan 12 04:34 spark-3.5.8-bin-hadoop3.tgz.2
Spark environment ready.


In [2]:
import sys
import glob
from google.colab import userdata

# Read from Colab Secrets
AWS_ACCESS_KEY = userdata.get("AWS_ACCESS_KEY")
AWS_SECRET_KEY = userdata.get("AWS_SECRET_KEY")
AWS_REGION     = userdata.get("AWS_REGION")
BUCKET_NAME    = userdata.get("BUCKET_NAME")
EIA_API_KEY    = userdata.get("EIA_API_KEY")

S3_INPUT = f"s3a://{BUCKET_NAME}/raw"

NAT_GAS_PATH  = f"{S3_INPUT}/natural_gas/"
NG_TRADE_PATH = f"{S3_INPUT}/natural_gas_trade/"
CRUDE_PATH    = f"{S3_INPUT}/crude_oil_imports/"

# S3 output
S3_BASE = f"s3a://{BUCKET_NAME}/processed"

S3_NG_PRODUCTION   = f"{S3_BASE}/natural_gas_production/"
S3_NG_INTL_TRADE   = f"{S3_BASE}/natural_gas_trade_international/"
S3_NG_INTERSTATE   = f"{S3_BASE}/natural_gas_trade_interstate/"
S3_CRUDE_BY_ORIGIN = f"{S3_BASE}/crude_oil_imports_by_state_country/"
S3_CRUDE_STATE     = f"{S3_BASE}/crude_oil_imports_by_state/"
S3_CRUDE_GRADE     = f"{S3_BASE}/crude_oil_imports_grade_breakdown/"

print("Config loaded.")
print(f"  Natural gas:   {NAT_GAS_PATH}")
print(f"  NG trade:      {NG_TRADE_PATH}")
print(f"  Crude imports: {CRUDE_PATH}")
print(f"  S3 bucket:     {BUCKET_NAME}")

Config loaded.
  Natural gas:   s3a://cs4266-eia-big-data-bucket/raw/natural_gas/
  NG trade:      s3a://cs4266-eia-big-data-bucket/raw/natural_gas_trade/
  Crude imports: s3a://cs4266-eia-big-data-bucket/raw/crude_oil_imports/
  S3 bucket:     cs4266-eia-big-data-bucket


## 1. Initialize Spark Session

In [4]:
from pyspark.sql import SparkSession

import os
print(os.environ.get('JAVA_HOME'))  # Should show the jdk-17 path


builder = (
    SparkSession.builder
    .appName('EIA Natural Gas & Crude Oil Processing')
    .config('spark.hadoop.validateOutputSpecs', False)
)

# Only wire up S3 if credentials are filled in — lets you run locally without AWS
if AWS_ACCESS_KEY and AWS_SECRET_KEY and AWS_REGION:
    builder = (
        builder
        .config('spark.hadoop.fs.s3a.impl',        'org.apache.hadoop.fs.s3a.S3AFileSystem')
        .config('spark.hadoop.fs.s3a.endpoint',    f's3.{AWS_REGION}.amazonaws.com')
        .config('spark.hadoop.fs.s3a.access.key',  AWS_ACCESS_KEY)
        .config('spark.hadoop.fs.s3a.secret.key',  AWS_SECRET_KEY)
    )
    print('S3 configured.')
else:
    print('No S3 credentials — local mode only (Sections 0–4).')

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')

/usr/lib/jvm/java-8-openjdk-amd64
S3 configured.
Spark 3.5.8 ready.


## 2. Load Raw Data

In [5]:
import boto3
import io
import pandas as pd

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION,
)

# Gets most recently modified CSV in the prefix
def fetch_latest_csv(prefix):
    """Return the most recently modified CSV under a given S3 prefix as a Spark DataFrame."""
    resp = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    if "Contents" not in resp:
        raise FileNotFoundError(f"No objects found under s3://{BUCKET_NAME}/{prefix}")

    csvs = [obj for obj in resp["Contents"] if obj["Key"].endswith(".csv")]
    if not csvs:
        raise FileNotFoundError(f"No CSV files found under s3://{BUCKET_NAME}/{prefix}")

    latest = max(csvs, key=lambda obj: obj["LastModified"])
    print(f"  Fetching: s3://{BUCKET_NAME}/{latest['Key']}")

    body = s3.get_object(Bucket=BUCKET_NAME, Key=latest["Key"])["Body"].read()
    pdf = pd.read_csv(io.BytesIO(body))
    return spark.createDataFrame(pdf)

print("Fetching raw CSVs from S3...")
raw_ng    = fetch_latest_csv("raw/natural_gas/")
raw_trade = fetch_latest_csv("raw/natural_gas_trade/")
raw_crude = fetch_latest_csv("raw/crude_oil_imports/")

print(f"\nNatural Gas Production raw rows : {raw_ng.count():>8,}")
print(f"Natural Gas Trade raw rows       : {raw_trade.count():>8,}")
print(f"Crude Oil Imports raw rows       : {raw_crude.count():>8,}")

Fetching raw CSVs from S3...
  Fetching: s3://cs4266-eia-big-data-bucket/raw/natural_gas/monthly/natural_gas_2026-01-01_to_2026-04-24.csv
  Fetching: s3://cs4266-eia-big-data-bucket/raw/natural_gas_trade/annual/natural_gas_trade_2023_to_2026.csv
  Fetching: s3://cs4266-eia-big-data-bucket/raw/crude_oil_imports/monthly/crude_oil_imports_2026-01-01_to_2026-04-24.csv

Natural Gas Production raw rows :       37
Natural Gas Trade raw rows       :    2,798
Crude Oil Imports raw rows       :    2,015


In [6]:
print("=== Natural Gas Production Schema ===")
raw_ng.printSchema()

print("\n=== Natural Gas Trade Schema ===")
raw_trade.printSchema()

print("\n=== Crude Oil Imports Schema ===")
raw_crude.printSchema()

=== Natural Gas Production Schema ===
root
 |-- period: string (nullable = true)
 |-- duoarea: string (nullable = true)
 |-- area-name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- product-name: string (nullable = true)
 |-- process: string (nullable = true)
 |-- process-name: string (nullable = true)
 |-- series: string (nullable = true)
 |-- series-description: string (nullable = true)
 |-- value: double (nullable = true)
 |-- units: string (nullable = true)


=== Natural Gas Trade Schema ===
root
 |-- period: long (nullable = true)
 |-- duoarea: string (nullable = true)
 |-- area-name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- product-name: string (nullable = true)
 |-- process: string (nullable = true)
 |-- process-name: string (nullable = true)
 |-- series: string (nullable = true)
 |-- series-description: string (nullable = true)
 |-- value: double (nullable = true)
 |-- units: string (nullable = true)


=== Crude Oil Imports Sch

## 3. Cleaning

### 3a. Natural Gas Production

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

ng_clean = (
    raw_ng
    # Remove rows with no reported production value
    .filter(F.col("value").isNotNull())
    # Keep state-level rows only (duoarea starts with "S")
    # Drops U.S. total (NUS), federal offshore (R3FM), and non-state entries
    .filter(F.col("duoarea").startswith("S"))
    # Keep only Marketed Production series (the primary production metric)
    .filter(F.col("process") == "VGM")
    # Parse period ("YYYY-MM") to a proper date (first of month)
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM"))
    # Cast value to double
    .withColumn("production_mmcf", F.col("value").cast("double"))
    # Rename and select relevant columns
    .withColumnRenamed("area-name", "state_name")
    .select("period", "duoarea", "state_name", "production_mmcf")
    # Drop duplicates that may arise from append-style file generation
    .dropDuplicates()
)

print(f"Natural Gas Production cleaned rows: {ng_clean.count():,}")
ng_clean.show(5)

Natural Gas Production cleaned rows: 34
+----------+-------+----------+---------------+
|    period|duoarea|state_name|production_mmcf|
+----------+-------+----------+---------------+
|2026-01-01|    SNE|    USA-NE|            NaN|
|2026-01-01|    SWY|    USA-WY|        79790.0|
|2026-01-01|    STX|     TEXAS|      1072107.0|
|2026-01-01|    SMO|    USA-MO|            NaN|
|2026-01-01|    SOK|    USA-OK|       246152.0|
+----------+-------+----------+---------------+
only showing top 5 rows



### 3b. Natural Gas Trade

In [8]:
# Map process codes to readable labels
PROCESS_LABELS = {
    "EEI": "Exports (Intl)",
    "IMI": "Imports (Intl)",
    "IMB": "Net Intl Movements",
    "MID": "Interstate Deliveries",
    "MIN": "Net Interstate Receipts",
    "MIE": "Net Intl + Interstate",
}
process_map = F.create_map(
    *[item for pair in [(F.lit(k), F.lit(v)) for k, v in PROCESS_LABELS.items()] for item in pair]
)

trade_clean = (
    raw_trade
    # Remove rows with null value
    .filter(F.col("value").isNotNull())
    # Keep state-level duoarea rows only
    .filter(F.col("duoarea").rlike(r"^S[A-Z]{2}"))
    # Keep only the 6 process codes of interest
    .filter(F.col("process").isin(list(PROCESS_LABELS.keys())))
    # Cast period to integer (annual data)
    .withColumn("year", F.col("period").cast(IntegerType()))
    # Add human-readable process label
    .withColumn("process_label", process_map[F.col("process")])
    # Rename and select
    .withColumnRenamed("area-name", "state_name")
    .withColumn("value_mmcf", F.col("value").cast("double"))
    .select("year", "duoarea", "state_name", "process", "process_label", "value_mmcf", "units")
    .dropDuplicates()
)

print(f"Natural Gas Trade cleaned rows: {trade_clean.count():,}")
trade_clean.show(5)

Natural Gas Trade cleaned rows: 2,014
+----+-------+----------+-------+------------------+----------+-----+
|year|duoarea|state_name|process|     process_label|value_mmcf|units|
+----+-------+----------+-------+------------------+----------+-----+
|2023|SMD-NTH|       THA|    EEI|    Exports (Intl)|    3728.0| MMCF|
|2023|STN-Z00|    USA-TN|    EEI|    Exports (Intl)|       0.0| MMCF|
|2023|SLA-NCH|       CHN|    EEI|    Exports (Intl)|   90227.0| MMCF|
|2023|SMO-Z00|    USA-MO|    IMB|Net Intl Movements|       0.0| MMCF|
|2023|STX-NIT|       ITA|    IMB|Net Intl Movements|  -69020.0| MMCF|
+----+-------+----------+-------+------------------+----------+-----+
only showing top 5 rows



### 3c. Crude Oil Imports

In [9]:
crude_clean = (
    raw_crude
    # Remove rows with null quantity
    .filter(F.col("quantity").isNotNull())
    # Filter to Refinery State level only to avoid double-counting
    # The raw data repeats each import at 7 geographic levels:
    # Port PADD, Port State, Port, Refinery, Refinery PADD, Refinery State, US total.
    # We keep "Refinery State" as the most useful state-level grain.
    .filter(F.col("destinationTypeName") == "Refinery State")
    # Keep only actual country origins (not OPEC/Non-OPEC aggregates)
    .filter(F.col("originType") == "CTY")
    # Parse period to date
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM"))
    # Cast quantity
    .withColumn("quantity_thousand_bbl", F.col("quantity").cast("double"))
    # Rename destination column for clarity
    .withColumnRenamed("destinationName", "refinery_state")
    .withColumnRenamed("originName",       "origin_country")
    .withColumnRenamed("gradeName",        "crude_grade")
    .select(
        "period", "origin_country", "originId",
        "refinery_state", "destinationId",
        "crude_grade", "quantity_thousand_bbl"
    )
    .dropDuplicates()
)

print(f"Crude Oil Imports cleaned rows: {crude_clean.count():,}")
crude_clean.show(5)

Crude Oil Imports cleaned rows: 110
+----------+--------------+--------+--------------+-------------+-----------+---------------------+
|    period|origin_country|originId|refinery_state|destinationId|crude_grade|quantity_thousand_bbl|
+----------+--------------+--------+--------------+-------------+-----------+---------------------+
|2026-01-01|       Nigeria|  CTY_NI|  Pennsylvania|        RS_PA|Light Sweet|                586.0|
|2026-01-01|        Canada|  CTY_CA|          Ohio|        RS_OH| Heavy Sour|               2770.0|
|2026-01-01|        Mexico|  CTY_MX|       Alabama|        RS_AL| Light Sour|                444.0|
|2026-01-01|         Ghana|  CTY_GH|    New Jersey|        RS_NJ|Light Sweet|                917.0|
|2026-01-01|        Guyana|  CTY_GY|    New Jersey|        RS_NJ| Light Sour|               1042.0|
+----------+--------------+--------+--------------+-------------+-----------+---------------------+
only showing top 5 rows



## 4. Analytics

### 4a. Natural Gas: State Production Over Time & % of US Total

In [10]:
# US monthly total (from cleaned state-level data)
us_ng_total = (
    ng_clean
    .groupBy("period")
    .agg(F.sum("production_mmcf").alias("us_total_mmcf"))
)

# Join state totals with US total, compute % share
ng_state_production = (
    ng_clean
    .join(us_ng_total, on="period", how="left")
    .withColumn(
        "pct_us_production",
        F.round(F.col("production_mmcf") / F.col("us_total_mmcf") * 100, 4)
    )
    .select(
        "period", "duoarea", "state_name",
        "production_mmcf", "us_total_mmcf", "pct_us_production"
    )
    .orderBy("period", F.col("production_mmcf").desc())
)

print(f"State production rows: {ng_state_production.count():,}")
ng_state_production.show(10)

State production rows: 34
+----------+-------+----------+---------------+-------------+-----------------+
|    period|duoarea|state_name|production_mmcf|us_total_mmcf|pct_us_production|
+----------+-------+----------+---------------+-------------+-----------------+
|2026-01-01|    SNV|    USA-NV|            NaN|          NaN|              NaN|
|2026-01-01|    SOR|    USA-OR|            NaN|          NaN|              NaN|
|2026-01-01|    SID|    USA-ID|            NaN|          NaN|              NaN|
|2026-01-01|    SAL|    USA-AL|            NaN|          NaN|              NaN|
|2026-01-01|    SIN|    USA-IN|            NaN|          NaN|              NaN|
|2026-01-01|    SVA|    USA-VA|            NaN|          NaN|              NaN|
|2026-01-01|    SNE|    USA-NE|            NaN|          NaN|              NaN|
|2026-01-01|    SMO|    USA-MO|            NaN|          NaN|              NaN|
|2026-01-01|    SSD|    USA-SD|            NaN|          NaN|              NaN|
|2026-01-01|  

### 4b. Natural Gas: International Trade Summary by State (Annual)

In [11]:
intl_processes = ["EEI", "IMI", "IMB"]

ng_intl_trade = (
    trade_clean
    .filter(F.col("process").isin(intl_processes))
    .groupBy("year", "duoarea", "state_name", "process", "process_label")
    .agg(F.sum("value_mmcf").alias("total_mmcf"))
    .orderBy("year", "state_name", "process")
)

# Pivot wider: one row per state-year, columns for each process
ng_intl_trade_wide = (
    ng_intl_trade
    .groupBy("year", "duoarea", "state_name")
    .pivot("process", intl_processes)
    .agg(F.first("total_mmcf"))
    .withColumnRenamed("EEI", "exports_mmcf")
    .withColumnRenamed("IMI", "imports_mmcf")
    .withColumnRenamed("IMB", "net_intl_mmcf")
    .orderBy("year", "state_name")
)

print(f"International trade rows (wide): {ng_intl_trade_wide.count():,}")
ng_intl_trade_wide.show(10)

International trade rows (wide): 361
+----+-------+----------+------------+------------+-------------+
|year|duoarea|state_name|exports_mmcf|imports_mmcf|net_intl_mmcf|
+----+-------+----------+------------+------------+-------------+
|2023|SMD-NAR|       ARG|      6962.0|        NULL|      -6962.0|
|2023|SGA-NAR|       ARG|      2343.0|        NULL|      -2343.0|
|2023|STX-NAR|       ARG|     21803.0|        NULL|     -21803.0|
|2023|SLA-NAR|       ARG|     45813.0|        NULL|     -45813.0|
|2023|STX-NBE|       BEL|     29960.0|        NULL|         NULL|
|2023|SLA-NBE|       BEL|     67056.0|        NULL|         NULL|
|2023|SLA-NBG|       BGD|     20585.0|        NULL|     -20585.0|
|2023|STX-NBG|       BGD|      3561.0|        NULL|      -3561.0|
|2023|SFL-NBF|       BHS|       499.0|        NULL|       -499.0|
|2023|SLA-NBR|       BRA|     26235.0|        NULL|     -26235.0|
+----+-------+----------+------------+------------+-------------+
only showing top 10 rows



### 4c. Natural Gas: Interstate Movements by State (Annual)

In [12]:
interstate_processes = ["MID", "MIN"]

ng_interstate = (
    trade_clean
    .filter(F.col("process").isin(interstate_processes))
    .groupBy("year", "duoarea", "state_name", "process", "process_label")
    .agg(F.sum("value_mmcf").alias("total_mmcf"))
    .orderBy("year", "state_name", "process")
)

ng_interstate_wide = (
    ng_interstate
    .groupBy("year", "duoarea", "state_name")
    .pivot("process", interstate_processes)
    .agg(F.first("total_mmcf"))
    .withColumnRenamed("MID", "interstate_delivered_mmcf")
    .withColumnRenamed("MIN", "net_interstate_received_mmcf")
    .orderBy("year", "state_name")
)

print(f"Interstate movements rows (wide): {ng_interstate_wide.count():,}")
ng_interstate_wide.show(10)

Interstate movements rows (wide): 552
+----+-------+----------+-------------------------+----------------------------+
|year|duoarea|state_name|interstate_delivered_mmcf|net_interstate_received_mmcf|
+----+-------+----------+-------------------------+----------------------------+
|2023|SOR-SCA|CALIFORNIA|                 738530.0|                   -738530.0|
|2023|SAZ-SCA|CALIFORNIA|                 990923.0|                   -901736.0|
|2023|SCO-SCA|CALIFORNIA|                      6.0|                        -6.0|
|2023|SCA-Z0S|CALIFORNIA|                 140036.0|                   2107544.0|
|2023|SWY-SCA|CALIFORNIA|                      3.0|                        -3.0|
|2023|SID-SCA|CALIFORNIA|                      1.0|                        -1.0|
|2023|SNV-SCA|CALIFORNIA|                 518117.0|                   -467268.0|
|2023|SMT-SCO|  COLORADO|                     NULL|                         2.0|
|2023|SCA-SCO|  COLORADO|                     NULL|                    

### 4d. Crude Oil: State Import Volume by Country of Origin (Monthly)

In [13]:
crude_by_origin = (
    crude_clean
    .groupBy("period", "refinery_state", "origin_country")
    .agg(F.sum("quantity_thousand_bbl").alias("total_thousand_bbl"))
    .orderBy("period", "refinery_state", F.col("total_thousand_bbl").desc())
)

print(f"Crude imports by state & origin rows: {crude_by_origin.count():,}")
crude_by_origin.show(10)

Crude imports by state & origin rows: 73
+----------+--------------+--------------+------------------+
|    period|refinery_state|origin_country|total_thousand_bbl|
+----------+--------------+--------------+------------------+
|2026-01-01|       Alabama|  Saudi Arabia|             479.0|
|2026-01-01|       Alabama|        Mexico|             444.0|
|2026-01-01|       Alabama|        Canada|             223.0|
|2026-01-01|       Alabama|        Kuwait|             149.0|
|2026-01-01|        Alaska|        Canada|             471.0|
|2026-01-01|    California|          Iraq|            6456.0|
|2026-01-01|    California|     Argentina|            4072.0|
|2026-01-01|    California|        Canada|            3792.0|
|2026-01-01|    California|        Brazil|            3680.0|
|2026-01-01|    California|        Guyana|            3074.0|
+----------+--------------+--------------+------------------+
only showing top 10 rows



### 4e. Crude Oil: Total State Imports Over Time & % of US Total

In [14]:
# US monthly total
us_crude_total = (
    crude_clean
    .groupBy("period")
    .agg(F.sum("quantity_thousand_bbl").alias("us_total_thousand_bbl"))
)

# State totals + % of US
crude_by_state = (
    crude_clean
    .groupBy("period", "refinery_state")
    .agg(F.sum("quantity_thousand_bbl").alias("total_thousand_bbl"))
    .join(us_crude_total, on="period", how="left")
    .withColumn(
        "pct_us_imports",
        F.round(F.col("total_thousand_bbl") / F.col("us_total_thousand_bbl") * 100, 4)
    )
    .orderBy("period", F.col("total_thousand_bbl").desc())
)

print(f"Crude imports by state rows: {crude_by_state.count():,}")
crude_by_state.show(10)

Crude imports by state rows: 24
+----------+--------------+------------------+---------------------+--------------+
|    period|refinery_state|total_thousand_bbl|us_total_thousand_bbl|pct_us_imports|
+----------+--------------+------------------+---------------------+--------------+
|2026-01-01|      Illinois|           40482.0|             200538.0|       20.1867|
|2026-01-01|         Texas|           32990.0|             200538.0|       16.4507|
|2026-01-01|    California|           24659.0|             200538.0|       12.2964|
|2026-01-01|      Oklahoma|           14864.0|             200538.0|        7.4121|
|2026-01-01|    New Jersey|           12304.0|             200538.0|        6.1355|
|2026-01-01|     Minnesota|           10078.0|             200538.0|        5.0255|
|2026-01-01|       Indiana|            9866.0|             200538.0|        4.9198|
|2026-01-01|    Washington|            9552.0|             200538.0|        4.7632|
|2026-01-01|   Mississippi|            6771.

### 4f. Crude Oil: Grade Breakdown per State (Monthly)

In [15]:
# State-month totals (for computing within-state grade share)
state_month_totals = (
    crude_clean
    .groupBy("period", "refinery_state")
    .agg(F.sum("quantity_thousand_bbl").alias("state_total_thousand_bbl"))
)

crude_grade_breakdown = (
    crude_clean
    .groupBy("period", "refinery_state", "crude_grade")
    .agg(F.sum("quantity_thousand_bbl").alias("grade_quantity_thousand_bbl"))
    .join(state_month_totals, on=["period", "refinery_state"], how="left")
    .withColumn(
        "pct_of_state_imports",
        F.round(F.col("grade_quantity_thousand_bbl") / F.col("state_total_thousand_bbl") * 100, 4)
    )
    .orderBy("period", "refinery_state", F.col("grade_quantity_thousand_bbl").desc())
)

print(f"Grade breakdown rows: {crude_grade_breakdown.count():,}")
crude_grade_breakdown.show(10)

Grade breakdown rows: 58
+----------+--------------+-----------+---------------------------+------------------------+--------------------+
|    period|refinery_state|crude_grade|grade_quantity_thousand_bbl|state_total_thousand_bbl|pct_of_state_imports|
+----------+--------------+-----------+---------------------------+------------------------+--------------------+
|2026-01-01|       Alabama|     Medium|                      479.0|                  1295.0|             36.9884|
|2026-01-01|       Alabama| Light Sour|                      444.0|                  1295.0|             34.2857|
|2026-01-01|       Alabama| Heavy Sour|                      372.0|                  1295.0|             28.7259|
|2026-01-01|        Alaska|     Medium|                      471.0|                   471.0|               100.0|
|2026-01-01|    California|     Medium|                    13071.0|                 24659.0|              53.007|
|2026-01-01|    California| Heavy Sour|                     751

## 5. Write Parquet to S3

**Requires a configured S3 bucket.** Fill in `example_config.py` first.  
All cells above this point run entirely locally without AWS credentials.

In [16]:
def write_parquet(df, path, label):
    """Write a DataFrame to S3 as snappy-compressed Parquet."""
    print(f"Writing {label} → {path} ...")
    df.write.mode("overwrite").option("compression", "snappy").parquet(path)
    print(f"Done")

write_parquet(ng_state_production,   S3_NG_PRODUCTION,   "Natural Gas State Production")
write_parquet(ng_intl_trade_wide,    S3_NG_INTL_TRADE,   "Natural Gas International Trade")
write_parquet(ng_interstate_wide,    S3_NG_INTERSTATE,   "Natural Gas Interstate Movements")
write_parquet(crude_by_origin,       S3_CRUDE_BY_ORIGIN, "Crude Imports by State & Country")
write_parquet(crude_by_state,        S3_CRUDE_STATE,     "Crude Imports by State")
write_parquet(crude_grade_breakdown, S3_CRUDE_GRADE,     "Crude Imports Grade Breakdown")

print("\nAll datasets written to S3 successfully.")

Writing Natural Gas State Production → s3a://cs4266-eia-big-data-bucket/processed/natural_gas_production/ ...
Done
Writing Natural Gas International Trade → s3a://cs4266-eia-big-data-bucket/processed/natural_gas_trade_international/ ...
Done
Writing Natural Gas Interstate Movements → s3a://cs4266-eia-big-data-bucket/processed/natural_gas_trade_interstate/ ...
Done
Writing Crude Imports by State & Country → s3a://cs4266-eia-big-data-bucket/processed/crude_oil_imports_by_state_country/ ...
Done
Writing Crude Imports by State → s3a://cs4266-eia-big-data-bucket/processed/crude_oil_imports_by_state/ ...
Done
Writing Crude Imports Grade Breakdown → s3a://cs4266-eia-big-data-bucket/processed/crude_oil_imports_grade_breakdown/ ...
Done

All datasets written to S3 successfully.


## 6. Stop Spark

In [17]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
